# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.3 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64, pickle
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
torch.set_num_threads(1)

In [6]:
TASK_ID = "task343"
CH = 10
H = W = 30
LOCAL_TASK_JSON = Path("/mnt/data/task343(1).json")
KAGGLE_TASK_JSON = Path(COMPETITION) / f"{TASK_ID}.json"
TASK_JSON = LOCAL_TASK_JSON if LOCAL_TASK_JSON.exists() else KAGGLE_TASK_JSON
WORKDIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT_DIR = WORKDIR / f"{TASK_ID}_onnx"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUBMISSION_PATH = WORKDIR / 'submission.zip'
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary.json"

with TASK_JSON.open() as f:
    task = json.load(f)

print('Task:', TASK_ID)
print('Task JSON:', TASK_JSON)
print('Splits:', {k: len(task.get(k, [])) for k in ['train', 'test', 'arc-gen']})

Task: task343
Task JSON: /kaggle/input/competitions/neurogolf-2026/task343.json
Splits: {'train': 3, 'test': 1, 'arc-gen': 262}


In [7]:

def grid_to_tensor_zero_padded(grid, h=H, w=W, ch=CH):
    """Convert an ARC grid to [1,10,30,30].
    Inside the real grid: one-hot, including background color 0.
    Outside the real grid: all-zero across channels, so padding is not background.
    """
    arr = np.zeros((1, ch, h, w), dtype=np.float32)
    gh, gw = len(grid), len(grid[0])
    for r in range(gh):
        for c in range(gw):
            arr[0, int(grid[r][c]), r, c] = 1.0
    return arr

def tensor_to_grid(y, h, w):
    return (y[0, :, :h, :w].argmax(axis=0).astype(np.int64)).tolist()

def active_from_x(x):
    return (x.sum(dim=1, keepdim=True) > 0.5).float()

RULE_DESCRIPTION = 'infer the smallest horizontal period matching the observed prefix and repeat every row across the active canvas'


In [8]:

def compose_output(active, color_masks):
    out = torch.zeros_like(active).expand(-1, 10, -1, -1).clone()
    occ = torch.zeros_like(active)
    for k, m in color_masks.items():
        m = (m > 0.5).float() * active
        out[:, k:k+1] = m
        if k != 0:
            occ = torch.clamp(occ + m, 0.0, 1.0)
    out[:, 0:1] = active * (1.0 - torch.clamp(occ, 0.0, 1.0))
    return out * active

class Task343HorizontalPeriod(nn.Module):
    def __init__(self):
        super().__init__()
        for p in range(1,16):
            idx=(torch.arange(30)%p).view(1,1,1,30).expand(1,10,30,30).long()
            self.register_buffer(f'idx_{p}',idx)
    def candidate(self,x,p,active):
        return torch.gather(x,3,getattr(self,f'idx_{p}'))*active
    def forward(self,x):
        active=(x.sum(dim=1,keepdim=True)>0.5).float()
        nonzero=x[:,1:10].sum(dim=1,keepdim=True)
        col_non=(nonzero.sum(dim=2,keepdim=True)>0.5).float()
        prefix=(torch.flip(torch.flip(col_non,dims=[3]).cumsum(dim=3),dims=[3])>0.5).float()
        out=torch.zeros_like(x)
        prev=torch.zeros((1,1,1,1), dtype=x.dtype, device=x.device)
        for p in range(1,16):
            cand=self.candidate(x,p,active)
            diff=torch.abs((cand-x)*prefix*active).sum()
            valid=(diff<0.5).float().view(1,1,1,1)*(1.0-prev)
            out=out+cand*valid
            prev=torch.clamp(prev+valid,0,1)
        return (out+x*(1.0-prev)*active)*active


MODEL_CLASS = Task343HorizontalPeriod

model = MODEL_CLASS().eval()
print(model)


Task343HorizontalPeriod()


In [9]:
dummy = torch.from_numpy(grid_to_tensor_zero_padded(task['test'][0]['input']))

torch.onnx.export(
    model,
    dummy,
    str(ONNX_PATH),
    input_names=['input'],
    output_names=['output'],
    opset_version=18,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)

print('Exported:', ONNX_PATH)
print('ONNX bytes:', ONNX_PATH.stat().st_size)

/tmp/ipykernel_16/1758635681.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported: /kaggle/working/task343_onnx/task343.onnx
ONNX bytes: 1143438


In [10]:
def vi_shape(vi):
    dims = []
    for d in vi.type.tensor_type.shape.dim:
        if d.dim_value:
            dims.append(int(d.dim_value))
        elif d.dim_param:
            dims.append(str(d.dim_param))
        else:
            dims.append(None)
    return dims

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
ops = collections.Counter(n.op_type for n in onnx_model.graph.node)
forbidden = {'Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function'}
empty_inputs = [n for n in onnx_model.graph.node for inp in n.input if inp == '']
bad_shapes = []
for vi in list(onnx_model.graph.input) + list(onnx_model.graph.output):
    shp = vi_shape(vi)
    if any(not isinstance(x, int) for x in shp):
        bad_shapes.append((vi.name, shp))

print('Input shape:', vi_shape(onnx_model.graph.input[0]))
print('Output shape:', vi_shape(onnx_model.graph.output[0]))
print('Ops:', dict(ops))
print('Forbidden ops present:', sorted(forbidden & set(ops)))
print('Empty optional inputs:', len(empty_inputs))
print('Non-static tensor shapes:', bad_shapes)
assert vi_shape(onnx_model.graph.input[0]) == [1, 10, 30, 30]
assert vi_shape(onnx_model.graph.output[0]) == [1, 10, 30, 30]
assert ONNX_PATH.stat().st_size < 1_400_000
assert not (forbidden & set(ops))
assert not bad_shapes

Input shape: [1, 10, 30, 30]
Output shape: [1, 10, 30, 30]
Ops: {'Constant': 96, 'ReduceSum': 18, 'Greater': 3, 'Cast': 18, 'Slice': 3, 'CumSum': 1, 'GatherElements': 15, 'Mul': 78, 'Sub': 30, 'Abs': 15, 'Less': 15, 'Reshape': 15, 'Add': 31, 'Clip': 15}
Forbidden ops present: []
Empty optional inputs: 0
Non-static tensor shapes: []


In [11]:
sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 1
sess_options.inter_op_num_threads = 1
sess = ort.InferenceSession(str(ONNX_PATH), sess_options=sess_options, providers=['CPUExecutionProvider'])

def validate_examples(examples):
    tensor_ok = 0
    grid_ok = 0
    outside_input_active_zero_ok = 0
    outside_expected_canvas_zero_ok = 0
    bad = []
    for i, ex in enumerate(examples):
        x = grid_to_tensor_zero_padded(ex['input'])
        y = sess.run(None, {'input': x})[0]
        exp = grid_to_tensor_zero_padded(ex['output'])
        pred_bin = (y > 0.5).astype(np.float32)
        if np.array_equal(pred_bin, exp):
            tensor_ok += 1
        else:
            bad.append(i)
        h, w = len(ex['output']), len(ex['output'][0])
        pred_grid = pred_bin[0, :, :h, :w].argmax(axis=0).astype(np.int64).tolist()
        if pred_grid == ex['output']:
            grid_ok += 1
        input_active = x.sum(axis=1, keepdims=True) > 0.5
        expected_active = exp.sum(axis=1, keepdims=True) > 0.5
        if np.all(np.abs(y * (~input_active)) < 1e-5):
            outside_input_active_zero_ok += 1
        if np.all(np.abs(y * (~expected_active)) < 1e-5):
            outside_expected_canvas_zero_ok += 1
    return {
        'tensor_exact_zero_padded': [tensor_ok, len(examples)],
        'grid_argmax_inside_output_canvas': [grid_ok, len(examples)],
        'outside_input_active_all_channels_zero': [outside_input_active_zero_ok, len(examples)],
        'outside_expected_output_canvas_all_channels_zero': [outside_expected_canvas_zero_ok, len(examples)],
        'bad_indices': bad[:10],
    }

def validate_split(split):
    return validate_examples(task[split])

rng = random.Random(0)
arcgen_indices = list(range(len(task.get('arc-gen', []))))
rng.shuffle(arcgen_indices)
holdout_n = max(1, int(math.ceil(0.60 * len(arcgen_indices)))) if arcgen_indices else 0
arcgen_holdout = [task['arc-gen'][i] for i in arcgen_indices[:holdout_n]]

summary = {
    'task_id': TASK_ID,
    'rule': RULE_DESCRIPTION,
    'onnx_path': str(ONNX_PATH),
    'onnx_size_bytes': ONNX_PATH.stat().st_size,
    'input_shape': vi_shape(onnx_model.graph.input[0]),
    'output_shape': vi_shape(onnx_model.graph.output[0]),
    'ops': dict(ops),
    'forbidden_ops': sorted(forbidden & set(ops)),
    'empty_optional_inputs': len(empty_inputs),
    'non_static_tensor_shapes': len(bad_shapes),
    'arc_gen_holdout_policy': 'deterministic random seed 0, 60% of arc-gen; full arc-gen also validated',
    'validation': {
        'train': validate_split('train'),
        'test': validate_split('test'),
        'arc-gen_60pct_holdout': validate_examples(arcgen_holdout),
        'arc-gen_full': validate_split('arc-gen'),
    },
}

print(json.dumps(summary, indent=2)[:6000])
with SUMMARY_PATH.open('w') as f:
    json.dump(summary, f, indent=2)

# Visible test and arc-gen are mandatory. Train pairs are also asserted except for task363, whose non-memorizing placement rule conflicts with two demonstration pairs.
_assert_train = True
for split_name, result in summary['validation'].items():
    if split_name == 'train' and not _assert_train:
        continue
    assert result['tensor_exact_zero_padded'][0] == result['tensor_exact_zero_padded'][1], split_name
    assert result['outside_expected_output_canvas_all_channels_zero'][0] == result['outside_expected_output_canvas_all_channels_zero'][1], split_name

{
  "task_id": "task343",
  "rule": "infer the smallest horizontal period matching the observed prefix and repeat every row across the active canvas",
  "onnx_path": "/kaggle/working/task343_onnx/task343.onnx",
  "onnx_size_bytes": 1143438,
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "ops": {
    "Constant": 96,
    "ReduceSum": 18,
    "Greater": 3,
    "Cast": 18,
    "Slice": 3,
    "CumSum": 1,
    "GatherElements": 15,
    "Mul": 78,
    "Sub": 30,
    "Abs": 15,
    "Less": 15,
    "Reshape": 15,
    "Add": 31,
    "Clip": 15
  },
  "forbidden_ops": [],
  "empty_optional_inputs": 0,
  "non_static_tensor_shapes": 0,
  "arc_gen_holdout_policy": "deterministic random seed 0, 60% of arc-gen; full arc-gen also validated",
  "validation": {
    "train": {
      "tensor_exact_zero_padded": [
        3,
        3
      ],
      "grid_argmax_inside_output_canvas": [
        3,
        3
      ],
      "outside_input_active

In [12]:
with zipfile.ZipFile(SUBMISSION_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')

print('Wrote:', SUBMISSION_PATH)
print('Zip contents:', zipfile.ZipFile(SUBMISSION_PATH).namelist())

Wrote: /kaggle/working/submission.zip
Zip contents: ['task343.onnx']
